Importing

In [1]:
import pandas as pd # for filing
from sklearn.model_selection import train_test_split # for train, test, split
from sklearn.ensemble import AdaBoostClassifier # Boosting model
from sklearn.tree import DecisionTreeClassifier # decison tree model
from sklearn.metrics import classification_report # for accuracy, f1score, precision, recall
from sklearn.ensemble import GradientBoostingClassifier # gradient boosting classifier
from xgboost import XGBClassifier #xg boosting classsifer

Dataset Link: https://www.kaggle.com/datasets/nimapourmoradi/raisin-binary-classification <br>
About Dataset: <br>
Area: Gives the number of pixels within the boundaries of the raisin. <br>
MajorAxisLength: Gives the length of the main axis, which is the longest line that can be drawn on the raisin <br>
MinorAxisLength: Gives the length of the small axis, which is the shortest line that can be drawn on the raisin. <br>
Eccentricity: It gives a measure of the eccentricity of the ellipse, which has the same moments as raisins. <br>
ConvexArea: Gives the number of pixels of the smallest convex shell of the region formed by the raisin. <br>
Extent: Gives the ratio of the region formed by the raisin to the total pixels in the bounding box. <br>
Perimeter: It measures the environment by calculating the distance between the boundaries of the raisin. <br>
Class: Kecimen and Besni raisin. <br>

Reaading Dataset

In [2]:
df = pd.read_csv("/content/Raisin_Dataset.csv")
df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,87524,442.246011,253.291155,0.819738,90546,0.758651,1184.040,Kecimen
1,75166,406.690687,243.032436,0.801805,78789,0.684130,1121.786,Kecimen
2,90856,442.267048,266.328318,0.798354,93717,0.637613,1208.575,Kecimen
3,45928,286.540559,208.760042,0.684989,47336,0.699599,844.162,Kecimen
4,79408,352.190770,290.827533,0.564011,81463,0.792772,1073.251,Kecimen


Unique binary classification

In [3]:
df["Class"].unique()

array(['Kecimen', 'Besni'], dtype=object)

Kecimen = 1
Besno = 0

In [4]:
df["Class"] = (df["Class"] == "Kecimen").astype(int)
df.tail()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
895,83248,430.077308,247.838695,0.817263,85839,0.668793,1129.072,0
896,87350,440.735698,259.293149,0.808629,90899,0.636476,1214.252,0
897,99657,431.706981,298.837323,0.721684,106264,0.741099,1292.828,0
898,93523,476.344094,254.176054,0.845739,97653,0.658798,1258.548,0
899,85609,512.081774,215.271976,0.907345,89197,0.632020,1272.862,0


Equal number of records

In [5]:
print(len(df[df["Class"] == 1]))
print(len(df[df["Class"] == 0]))

450
450


Getting X, Y splits

In [6]:
x = df[df.columns[:-1]]
y = df[df.columns[-1]]

Test, Train, Split

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

Model 1

In [8]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # small week learners
    n_estimators=50,   # number of small weak learners
    learning_rate=1.0, # step size
    random_state=42
)

Training and Prediction

In [9]:
ada.fit(x_train, y_train)
prediction = ada.predict(x_test)

Model classification report

In [10]:
print(classification_report(y_test, prediction))
print("Test score: ", ada.score(x_test, y_test))
print("Train score: ", ada.score(x_train, y_train))
# very small difference between test and train scores meaning there is no overfitting

              precision    recall  f1-score   support

           0       0.84      0.85      0.84        86
           1       0.86      0.85      0.86        94

    accuracy                           0.85       180
   macro avg       0.85      0.85      0.85       180
weighted avg       0.85      0.85      0.85       180

Test score:  0.85
Train score:  0.8694444444444445


Model 2

In [11]:
GBC = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3
)

Training and Prediction

In [12]:
GBC.fit(x_train, y_train)
prediction = GBC.predict(x_test)

Model classification report

In [13]:
print(classification_report(y_test, prediction))
print("Test score: ", GBC.score(x_test, y_test))
print("Train score: ", GBC.score(x_train, y_train))

              precision    recall  f1-score   support

           0       0.86      0.84      0.85        86
           1       0.85      0.87      0.86        94

    accuracy                           0.86       180
   macro avg       0.86      0.85      0.86       180
weighted avg       0.86      0.86      0.86       180

Test score:  0.8555555555555555
Train score:  0.9694444444444444


Model 3

In [14]:
XGB = XGBClassifier(
    n_estimators=2,
    max_depth=2,
    learning_rate=1,
    objective='binary:logistic'
)

Training and Prediction

In [16]:
XGB.fit(x_train, y_train)
prediction = XGB.predict(x_test)

Model Classification report

In [17]:
print(classification_report(y_test, prediction))
print("Test score: ", XGB.score(x_test, y_test))
print("Train score: ", XGB.score(x_train, y_train))

              precision    recall  f1-score   support

           0       0.83      0.86      0.85        86
           1       0.87      0.84      0.85        94

    accuracy                           0.85       180
   macro avg       0.85      0.85      0.85       180
weighted avg       0.85      0.85      0.85       180

Test score:  0.85
Train score:  0.8736111111111111


Tuning Hyper parameters of gradient boosting

In [18]:
estimators = [50, 100, 150, 200]
learning_rate = [0.01, 0.1, 0.5, 0.9]
max_depth = [1, 3, 7, 12]

In [22]:
best_score = 0
for estimator in estimators:
  for learn_rate in learning_rate:
    for depth in max_depth:

      print("-"*15)
      print("Estimator: ", estimator)
      print("Learning Rate: ", learn_rate)
      print("Max Depth: ", depth)
      print("-"*15)

      GBC = GradientBoostingClassifier(
          n_estimators=estimator,
          learning_rate=learn_rate,
          max_depth=depth
      )

      GBC.fit(x_train, y_train)
      prediction = GBC.predict(x_test)

      print(classification_report(y_test, prediction))
      print("Train score: ", GBC.score(x_train, y_train))
      print("Test score: ", GBC.score(x_test, y_test))

      if best_score < GBC.score(x_test, y_test):
        best_score = GBC.score(x_test, y_test)
        best_estimator = estimator
        best_learning_rate = learn_rate
        best_max_depth = depth

print("-"*15)
print("Best Score: ", best_score)
print("Best Estimator: ", best_estimator)
print("Best Learning Rate: ", best_learning_rate)
print("Best Max Depth: ", best_max_depth)
print("-"*15)

---------------
Estimator:  50
Learning Rate:  0.01
Max Depth:  1
---------------
              precision    recall  f1-score   support

           0       0.84      0.85      0.84        86
           1       0.86      0.85      0.86        94

    accuracy                           0.85       180
   macro avg       0.85      0.85      0.85       180
weighted avg       0.85      0.85      0.85       180

Train score:  0.8694444444444445
Test score:  0.85
---------------
Estimator:  50
Learning Rate:  0.01
Max Depth:  3
---------------
              precision    recall  f1-score   support

           0       0.85      0.85      0.85        86
           1       0.86      0.86      0.86        94

    accuracy                           0.86       180
   macro avg       0.86      0.86      0.86       180
weighted avg       0.86      0.86      0.86       180

Train score:  0.8916666666666667
Test score:  0.8555555555555555
---------------
Estimator:  50
Learning Rate:  0.01
Max Depth:  7
